# Section 2 - Preprocessing

We use the problems found in `Exploration_notebook.ipynb` to prepare the inputs.
The helpers below follow the Week 3 and Week 4 class notebooks.

## 1. Load the data

Keep `ID` as the index. It is only needed to match purchases to predictions.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

project_folder = Path.cwd()
if not (project_folder / "data").exists():
    project_folder = project_folder.parent

data_folder = project_folder / "data"
train = pd.read_csv(data_folder / "train.csv", index_col="ID")

target = "repeat_purchase_90d"
X = train.drop(columns=target)
y = train[target].astype(int)

assert train[target].notna().all()
assert set(train[target].unique()) == {0, 1}

Exploration found 2,880 non-repeat purchases and 3,654 repeat purchases. We keep the class balance.

The numeric group excludes `customer_id` and `random_noise`. The six categorical columns have no numeric order.

In [2]:
categorical_columns = [
    "payment_type", "salesperson", "commercial_zone",
    "transport_method", "warehouse", "document_series",
]
numeric_columns = [c for c in X.select_dtypes("number").columns
                   if c not in ["customer_id", "random_noise"]]

## 2. Clean invalid values

Negative amounts, counts and durations become missing. The billion-sized values in `prior_mean_purchase_value` and `spend_365d` also become missing.

Other large purchases stay. We use `log1p` later to reduce their scale.

In [3]:
# Adapted from Week 3: copy the frame and mask invalid cells.
def clean_values(data):
    cleaned = data.copy()
    values = cleaned[numeric_columns].replace([np.inf, -np.inf], np.nan)
    cleaned[numeric_columns] = values.mask(values < 0, np.nan)

    error_columns = ["prior_mean_purchase_value", "spend_365d"]
    cleaned[error_columns] = cleaned[error_columns].mask(
        cleaned[error_columns] >= 1_000_000_000, np.nan
    )
    return cleaned

In [4]:
# Check: valid values stay; negative, infinite and billion-sized values become NaN.
example = X.iloc[:4].copy()
example.index = ["valid", "negative", "infinite", "sentinel"]
example["days_since_previous_purchase"] = [10, -4, np.inf, 30]
example["prior_mean_purchase_value"] = [100, 200, 300, 1_000_000_000]
columns = ["days_since_previous_purchase", "prior_mean_purchase_value"]
result = clean_values(example)
display(pd.concat({"before": example[columns], "after": result[columns]}, axis=1))

assert result.loc["valid", "days_since_previous_purchase"] == 10
assert result.loc[["negative", "infinite"], "days_since_previous_purchase"].isna().all()
assert pd.isna(result.loc["sentinel", "prior_mean_purchase_value"])

# Count the replacements in the actual data too.
cleaned = clean_values(X)
replaced = (X[numeric_columns].notna() & cleaned[numeric_columns].isna()).sum()
display(replaced[replaced > 0].to_frame("values replaced"))

before                            \
         days_since_previous_purchase prior_mean_purchase_value   
valid                            10.0                       100   
negative                         -4.0                       200   
infinite                          inf                       300   
sentinel                         30.0                1000000000   

                                after                            
         days_since_previous_purchase prior_mean_purchase_value  
valid                            10.0                     100.0  
negative                          NaN                     200.0  
infinite                          NaN                     300.0  
sentinel                         30.0                       NaN

,values replaced
days_since_previous_purchase,16
prior_mean_purchase_value,32
prior_mean_gap_days,16
spend_365d,30


## 3. Missing history and extra features

We retain all 6,534 labelled rows; the supplied target is non-missing and binary, and is defined over the 90 days after each purchase. This includes 65 almost-empty rows without purchase dates and 10 rows identical across all non-ID fields. The undated rows cannot enter chronological validation, but they are included in the final all-row model fit. We keep matching records: identical fields and distinct IDs do not establish whether they represent separate or duplicate purchases.

There are also 603 first purchases. Missing history on these rows is expected. We add `first_purchase`, `history_unknown` and numeric missing-value flags before filling the gaps.

We also add net value per detail and remove spaces around category labels.

In [5]:
def add_features(data):
    prepared = clean_values(data)

    for column in numeric_columns:
        prepared[column + "_missing"] = prepared[column].isna().astype(int)

    prepared["first_purchase"] = (prepared["prior_purchase_count"] == 0).astype(int)
    prepared["history_unknown"] = prepared["prior_purchase_count"].isna().astype(int)

    details = prepared["N_ Detalhes"].where(prepared["N_ Detalhes"] > 0)
    prepared["net_value_per_detail"] = prepared["Total Liquido"] / details

    for column in categorical_columns:
        labels = prepared[column].astype("string").str.strip().replace("", pd.NA)
        prepared[column] = labels.to_numpy(dtype=object, na_value=np.nan)

    return prepared.drop(columns=["customer_id", "purchase_date", "random_noise"])

In [6]:
# Check first purchases, unknown history, zero details and spaces in labels.
example = X.iloc[:3].copy()
example.index = ["first", "with_history", "unknown"]
example["prior_purchase_count"] = [0, 2, np.nan]
example["Total Liquido"] = [100, 120, 50]
example["N_ Detalhes"] = [2, 0, np.nan]
example["payment_type"] = ["  Card  ", "", None]
result = add_features(example)

display(result[["first_purchase", "history_unknown", "prior_purchase_count_missing",
                "N_ Detalhes_missing", "net_value_per_detail", "payment_type"]])

# Expected: only the first row is a first purchase; only the last has unknown history.
assert result["first_purchase"].tolist() == [1, 0, 0]
assert result["history_unknown"].tolist() == [0, 0, 1]
assert result.loc["first", "net_value_per_detail"] == 50
assert result.loc[["with_history", "unknown"], "net_value_per_detail"].isna().all()
assert result.loc["first", "payment_type"] == "Card"
assert result.loc[["with_history", "unknown"], "payment_type"].isna().all()
assert not {"customer_id", "purchase_date", "random_noise"}.intersection(result.columns)

,first_purchase,history_unknown,prior_purchase_count_missing,N_ Detalhes_missing,net_value_per_detail,payment_type
first,1,0,0,0,50.0,Card
with_history,0,0,0,0,NaN,NaN
unknown,0,1,1,1,NaN,NaN


We leave out `clientes.csv`. Exploration found empty fields and many identifying labels. We also do not know when the customer information was last updated.

`customer_id` is a join key. `purchase_date` is used for splitting only. We leave out `random_noise` for now; feature selection checks that choice.

## 4. Split by date

Use roughly the last 20% of dated purchases for validation. Keep purchases on the same date together.

Leave a 90-day gap so the training targets are known before validation starts. Rows without dates are left out of this check and included again in the final fit.

In [7]:
purchase_dates = pd.to_datetime(X["purchase_date"], errors="coerce")
sorted_dates = purchase_dates.dropna().sort_values()
validation_start = sorted_dates.iloc[int(len(sorted_dates) * 0.80)]

train_rows = purchase_dates < validation_start - pd.Timedelta(days=90)
validation_rows = purchase_dates >= validation_start
gap_rows = purchase_dates.notna() & ~train_rows & ~validation_rows

X_train = X.loc[train_rows].copy()
X_val = X.loc[validation_rows].copy()
y_train = y.loc[train_rows]
y_val = y.loc[validation_rows]

assert not X_train.empty and not X_val.empty
assert purchase_dates[train_rows].max() + pd.Timedelta(days=90) < validation_start

print("Validation starts:", validation_start.date())
print("Training rows:", len(X_train))
print("Validation rows:", len(X_val))
print("Gap rows:", gap_rows.sum())
print("Rows without a date:", purchase_dates.isna().sum())

Validation starts: 2023-06-17
Training rows: 5024
Validation rows: 1298
Gap rows: 147
Rows without a date: 65


The split gives 5,024 training rows and 1,298 validation rows. There are 147 rows in the gap and 65 without dates.

A customer can appear in both periods. This checks future purchases, including those of existing customers. Exploration already used the full training data, so validation is not completely untouched.

## 5. Log transformation

Apply `log1p` to amounts, counts and durations, including value per detail. Leave the 0/1 flags unchanged.

We keep the log transformation before median filling, as in the feature-selection and model notebooks. This is the main order difference from Week 4.

In [8]:
log_columns = numeric_columns + ["net_value_per_detail"]


def prepare_features(data):
    prepared = add_features(data)
    prepared[log_columns] = np.log1p(prepared[log_columns])
    return prepared

In [9]:
# Check: 0, 9 and 99 become about 0, 2.303 and 4.605. Flags stay 0/1.
example = X_train.iloc[:3].copy()
example["Total Liquido"] = [0, 9, 99]
example["N_ Detalhes"] = 1
example["prior_purchase_count"] = [0, 1, 2]
before = add_features(example)
after = prepare_features(example)
columns = ["Total Liquido", "net_value_per_detail", "first_purchase"]
display(pd.concat({"before": before[columns], "after": after[columns]}, axis=1).round(3))

np.testing.assert_allclose(after["Total Liquido"], [0, 2.302585093, 4.605170186])
assert after["first_purchase"].tolist() == [1, 0, 0]

model_numeric_columns = prepare_features(X_train).select_dtypes("number").columns.tolist()
print("Numeric columns, including flags:", len(model_numeric_columns))
print("Categorical columns:", len(categorical_columns))

before                                             after  \
         Total Liquido net_value_per_detail first_purchase Total Liquido   
ID                                                                         
10000001             0                  0.0              1         0.000   
10000002             9                  9.0              0         2.303   
10000003            99                 99.0              0         4.605   

                                              
         net_value_per_detail first_purchase  
ID                                            
10000001                0.000              1  
10000002                2.303              0  
10000003                4.605              0

Numeric columns, including flags: 59
Categorical columns: 6


## 6. Fill missing values

Use the training median for numeric columns and `__MISSING__` for categories. Keep the flags added earlier.

This is the `fill_missing` helper from Week 4, with shorter comments. Each imputer fits on the training part only.

In [10]:
def fill_missing(train, test, plan):
    """Fit each imputer on train and fill both parts."""
    train, test = train.copy(), test.copy()
    for columns, imputer in plan:
        # sklearn needs np.nan rather than pd.NA.
        blocks = [half[columns].astype(object).fillna(np.nan).infer_objects()
                  for half in (train, test)]
        imputer.fit(blocks[0])
        train[columns], test[columns] = [imputer.transform(block) for block in blocks]
    return train, test

## 7. Encode and scale

Use one-hot encoding for categories. Unknown labels get zeros in that category's block. We keep all levels to match the other project notebooks; Week 4 drops the first level.

Use `StandardScaler` for numeric columns, including flags. One-hot columns stay at 0/1. Set `scale=False` for trees.

`fold_matrices` follows Week 4. It creates fresh imputers, an encoder and a scaler for each split. Its `test` argument can be validation data or the Kaggle test data.

In [11]:
# Adapted from Week 4's fold_matrices for this dataset.
def fold_matrices(train, test, scale=True):
    """Prepare one split, fitting only on its training rows."""
    train = prepare_features(train)
    test = prepare_features(test)

    plan = [
        (model_numeric_columns, SimpleImputer(strategy="median", keep_empty_features=True)),
        (categorical_columns, SimpleImputer(strategy="constant", fill_value="__MISSING__")),
    ]
    train, test = fill_missing(train, test, plan)

    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    cat_train = encoder.fit_transform(train[categorical_columns])
    cat_test = encoder.transform(test[categorical_columns])

    num_train = train[model_numeric_columns].to_numpy(dtype=float)
    num_test = test[model_numeric_columns].to_numpy(dtype=float)
    if scale:
        scaler = StandardScaler().fit(num_train)
        num_train = scaler.transform(num_train)
        num_test = scaler.transform(num_test)

    names = (["numeric__" + c for c in model_numeric_columns]
             + ["categorical__" + c for c in encoder.get_feature_names_out(categorical_columns)])
    return (np.hstack([num_train, cat_train]),
            np.hstack([num_test, cat_test]),
            np.asarray(names))

## 8. Check the result

Try `fold_matrices` on the training and validation split. We should get 5,024 and 1,298 rows, both with 164 columns and no missing or infinite values.

In [12]:
X_train_prepared, X_val_prepared, column_names = fold_matrices(X_train, X_val)

print("Training:", X_train_prepared.shape)
print("Validation:", X_val_prepared.shape)

assert len(X_train_prepared) == len(X_train)
assert len(X_val_prepared) == len(X_val)
assert X_train_prepared.shape[1] == X_val_prepared.shape[1] == len(column_names)
assert np.isfinite(X_train_prepared).all()
assert np.isfinite(X_val_prepared).all()

prepared_preview = pd.DataFrame(
    X_train_prepared[:3], columns=column_names, index=X_train.index[:3]
)
display(prepared_preview.iloc[:, :5])

Training: (5024, 164)
Validation: (1298, 164)


,numeric__Total a Pagar,numeric__Total Bruto,numeric__Total Liquido,numeric__Total Imp_,numeric__Total Portes
ID,,,,,
10000001,0.674879,0.606556,0.663172,0.739154,-0.455063
10000002,-2.627660,-2.476191,-2.616205,-2.146727,-0.455063
10000003,0.576436,0.476806,0.565005,0.646913,-0.455063


Both parts have 164 columns and no missing or infinite values. A negative scaled value means below the training mean.

This checks preparation only. Model performance is assessed later.

## 9. Prepare the full training and test data

Run the same helper on all training rows and `test.csv`. All fitted values still come from training only. Keep the IDs in their original order.

In [13]:
test = pd.read_csv(data_folder / "test.csv", index_col="ID")
assert set(test.columns) == set(X.columns)
assert train.index.is_unique and test.index.is_unique

X_all_prepared, X_test_prepared, final_column_names = fold_matrices(X, test[X.columns])
train_ids = X.index.copy()
test_ids = test.index.copy()

print("Full training:", X_all_prepared.shape)
print("Test:", X_test_prepared.shape)

assert len(X_all_prepared) == len(train_ids)
assert len(X_test_prepared) == len(test_ids)
assert X_all_prepared.shape[1] == X_test_prepared.shape[1] == len(final_column_names)
assert np.isfinite(X_all_prepared).all()
assert np.isfinite(X_test_prepared).all()

Full training: (6534, 173)
Test: (986, 173)


We keep all 6,534 training rows and 986 test rows. Both have 173 prepared columns. There are more categories in the full training period than in the earlier split.

Feature selection and modelling use the same rules inside a `Pipeline`, fitted again in each fold. They do not use these already-prepared full-data arrays for validation.

Class code used:

- Week 3 - cleaning: mask invalid cells with missing values.
- Week 4 - feature work: `fill_missing`, `fold_matrices`, one-hot encoding and scaling.

`fill_missing` is copied from Week 4, with shorter comments. The other helpers are adapted for this project and have examples above.